# CPML validation after the high-side boundary correction

This notebook reproduces the two scenarios used by `tests/test_014_cpml.py` and is intended to explain why the stored impedance regression changes after the dual-grid and high-side boundary corrections.

It checks two separate questions:

1. **Does the CPML still absorb correctly?**  
   A Gaussian packet is propagated towards the high-\(z\) CPML boundary. The forward and reflected fields and the frequency-dependent reflection factor are inspected.

2. **Why does the stored wake/impedance regression change?**  
   The current WAKIS longitudinal impedance is compared against the previous regression samples stored in `test_014_cpml.py`.

For the wake simulation, the first dominant transverse PEC resonance is also calculated from the STL bounds. With PEC boundaries in \(x\) and \(y\), the first on-axis-coupled TM mode is $\mathrm{TM}_{110}$,

$$
f_{110} =
\frac{c}{2}
\sqrt{
\left(\frac{1}{a}\right)^2 +
\left(\frac{1}{b}\right)^2
},
$$

with $a=x_{\max}-x_{\min}$ and $b=y_{\max}-y_{\min}$.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from scipy.constants import c
from tqdm import tqdm

import wakis

## Repository paths

The notebook is intended to live in `examples/`. The code below also works when the notebook kernel starts in the repository root.

In [ ]:
repo_root = Path.cwd()

if repo_root.name in ("examples", "notebooks"):
    repo_root = repo_root.parent

stl_file = repo_root / "tests" / "stl" / "001_cubic_cavity.stl"
ez_file = repo_root / "tests" / "014_Ez_diagnostic.h5"

print(f"Repository root: {repo_root}")
print(f"STL file:        {stl_file}")

## Previous impedance regression data

These are the stored `Zabs` samples from `tests/test_014_cpml.py`. They represent the solver behaviour before the dual-grid and high-side boundary corrections.

In [ ]:
Zabs_reference = np.array(
    [
        3.47693052e00,
        5.97952135e00,
        2.23594356e00,
        1.23041419e01,
        1.22201430e01,
        1.28218589e01,
        2.29114040e01,
        1.75525664e01,
        2.56345387e01,
        3.21612781e01,
        2.44292230e01,
        4.00746924e01,
        3.94104665e01,
        3.53968300e01,
        5.53200893e01,
        4.47282125e01,
        5.19363120e01,
        7.02880426e01,
        4.96917143e01,
        7.45682232e01,
        8.35969888e01,
        5.88591427e01,
        1.03503345e02,
        9.35392058e01,
        8.00242393e01,
        1.39037339e02,
        9.83042826e01,
        1.21894553e02,
        1.81927304e02,
        9.80347377e01,
        1.96021453e02,
        2.34397155e02,
        1.08701529e02,
        3.30726630e02,
        3.04232502e02,
        2.10965648e02,
        6.36343822e02,
        4.32886760e02,
        7.58951826e02,
        2.33175587e03,
        3.31244192e03,
        2.97231161e03,
        1.52717039e03,
        1.95922650e02,
        7.93183689e02,
        5.28075340e02,
        2.83064438e02,
        5.13982844e02,
        3.23441265e02,
        3.25819911e02,
    ]
)

# 1. Direct CPML reflection test

A Gaussian packet is launched towards the high-\(z\) CPML boundary. The setup is identical to the direct reflection test in `test_014_cpml.py`:

- periodic boundaries in \(x\) and \(y\);
- PEC at low \(z\);
- CPML at high \(z\);
- \(n_\mathrm{PML}=8\).

The test uses two reflection criteria:

$$
R_\mathrm{pulse}
=
\left(
\frac{\max |E_x^\mathrm{back}|}
     {\max |E_x^\mathrm{for}|}
\right)^2
\le 1.1\times10^{-6},
$$

and

$$
R(f)
=
\left|
\frac{E_x^\mathrm{back}(f)}
     {E_x^\mathrm{for}(f)}
\right|^2
\le 10^{-4}
$$

for frequencies up to \(6.66\,\mathrm{GHz}\).

In [ ]:
xmin, xmax = -1.0, 1.0
ymin, ymax = -1.0, 1.0
zmin, zmax = 0.0, 1.0

Nx, Ny = 8, 8
Nz = 200

grid_reflection = wakis.GridFIT3D(
    xmin,
    xmax,
    ymin,
    ymax,
    zmin,
    zmax,
    Nx,
    Ny,
    Nz,
)

solver_reflection = wakis.SolverFIT3D(
    grid_reflection,
    use_stl=False,
    use_gpu=False,
    bg=[1.0, 1.0, 0.0],
    bc_low=["periodic", "periodic", "pec"],
    bc_high=["periodic", "periodic", "cpml"],
    n_pml=8,
    kappa_max=5,
    alpha_max=0.05,
    sigma_factor=1,
    pml_exp=4,
    dtype=np.float32,
)

amplitude = 1.0

gaussian_packet = wakis.sources.GaussianPacket(
    xs=slice(0, Nx),
    ys=slice(0, Ny),
    sigmaz=15e-3,
    sigmaxy=100.0,
    amplitude=amplitude,
)

Nt = int(gaussian_packet.tinj + 2.0 * (zmax - zmin) / c / solver_reflection.dt)

forward = int((gaussian_packet.tinj + 0.5 * (zmax - zmin)) / c / solver_reflection.dt)

backward = int((gaussian_packet.tinj + 1.5 * (zmax - zmin)) / c / solver_reflection.dt)

Exfor = None
Exback = None

for n in tqdm(range(Nt)):
    gaussian_packet.update(solver_reflection, n * solver_reflection.dt)
    solver_reflection.one_step()

    if n == forward:
        Exfor = solver_reflection.E[
            Nx // 2,
            Ny // 2,
            : -solver_reflection.n_pml,
            "x",
        ].copy()

    if n == backward:
        Exback = solver_reflection.E[
            Nx // 2,
            Ny // 2,
            : -solver_reflection.n_pml,
            "x",
        ].copy()

if Exfor is None or Exback is None:
    raise RuntimeError("Forward/backward field snapshots were not recorded.")

reflection_factor = (np.abs(Exback).max() / np.abs(Exfor).max()) ** 2

print(f"Reflection factor: {reflection_factor:.6e}")
print("Test limit:        1.100000e-06")
print("Status:            " + ("PASS" if reflection_factor <= 1.1e-6 else "FAIL"))

### Forward and reflected Gaussian packet

In [ ]:
z_plot = solver_reflection.z[: -solver_reflection.n_pml]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    z_plot,
    Exfor,
    linewidth=1.5,
    label="Forward wave",
)

ax.plot(
    z_plot,
    Exback,
    linewidth=1.5,
    label="Reflected wave",
)

ax.set_xlabel("z [m]")
ax.set_ylabel(r"$E_x$")
ax.set_title("CPML Gaussian-packet reflection")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

### Frequency-dependent CPML reflection

In [ ]:
t = solver_reflection.z[: -solver_reflection.n_pml] / c

Sfor = np.abs(np.fft.fft(Exfor))
Sback = np.abs(np.fft.fft(Exback))

with np.errstate(divide="ignore", invalid="ignore"):
    S = (Sback / Sfor) ** 2

f = np.fft.fftfreq(len(t), d=t[1] - t[0])

mask = (0 <= f) & (f <= 6.66e9) & np.isfinite(S)

Smax = np.max(S[mask])

print(f"Maximum spectral reflection: {Smax:.6e}")
print("Test limit:                  1.000000e-04")
print("Status:                      " + ("PASS" if Smax <= 1e-4 else "FAIL"))

fig, ax = plt.subplots(figsize=(10, 5))

ax.semilogy(
    f[mask] / 1e9,
    S[mask],
    linewidth=1.5,
    label="Current CPML",
)

ax.axhline(
    1e-4,
    linestyle="--",
    linewidth=1.0,
    label="Test limit",
)

ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel(r"$|E_\mathrm{back}/E_\mathrm{for}|^2$")
ax.set_title("CPML reflection spectrum")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

# 2. CPML wake and longitudinal impedance

The second simulation reproduces the wake setup from `test_014_cpml.py`.

The transverse boundaries are PEC,

$$
x,y:\quad \mathrm{PEC}/\mathrm{PEC},
$$

while both longitudinal boundaries are CPML,

$$
z:\quad \mathrm{CPML}/\mathrm{CPML}.
$$

The current impedance is compared with the stored regression samples. Since the high-side PEC correction changes the effective transverse discretisation, a shift of transverse cavity resonances is expected even though the CPML itself remains unchanged in purpose.

In [ ]:
Nx = 50
Ny = 50
Nz = 150

surf = pv.read(str(stl_file))

stl_solids = {"cavity": str(stl_file)}
stl_materials = {"cavity": "vacuum"}

xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds

a = xmax - xmin
b = ymax - ymin
d = zmax - zmin

f110 = 0.5 * c * np.sqrt((1.0 / a) ** 2 + (1.0 / b) ** 2)

print("Cavity bounds:")
print(f"  a = {a:.6g} m")
print(f"  b = {b:.6g} m")
print(f"  d = {d:.6g} m")
print(f"Analytical transverse TM110 reference: {f110 / 1e9:.6f} GHz")

grid_wake = wakis.GridFIT3D(
    xmin,
    xmax,
    ymin,
    ymax,
    zmin,
    zmax,
    Nx,
    Ny,
    Nz,
    stl_solids=stl_solids,
    stl_materials=stl_materials,
    verbose=2,
)

beta = 1.0
sigmaz = 18.5e-3 * beta
q = 1e-9

xs = 0.0
ys = 0.0
xt = 0.0
yt = 0.0

skip_cells = 8
wakelength = 1.0

wake = wakis.WakeSolver(
    wakelength=wakelength,
    q=q,
    sigmaz=sigmaz,
    beta=beta,
    xsource=xs,
    ysource=ys,
    xtest=xt,
    ytest=yt,
    save=False,
    Ez_file=str(ez_file),
    skip_cells=skip_cells,
    verbose=2,
)

solver_wake = wakis.SolverFIT3D(
    grid_wake,
    wake,
    bc_low=["pec", "pec", "cpml"],
    bc_high=["pec", "pec", "cpml"],
    use_stl=True,
    bg="pec",
    dtype=np.float32,
    use_gpu=False,
    verbose=2,
    n_pml=4,
)

solver_wake.wakesolve(
    wakelength=wakelength,
    save_J=False,
)

if ez_file.exists():
    ez_file.unlink()

## Current impedance vs. previous regression

The solid curve is the fully resolved current WAKIS result. The dashed curve with markers contains the previously stored regression samples.

The dashed vertical line marks the analytical transverse TM$_{110}$ reference frequency calculated directly from the STL bounds.

In [ ]:
Z_current = np.abs(wake.Z)
f_current = wake.f

Z_sampled = Z_current[::20]
f_reference = f_current[::20]

if len(Z_sampled) != len(Zabs_reference):
    raise RuntimeError(
        "Current wake.Z[::20] length does not match the stored CPML regression data."
    )

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    f_current / 1e9,
    Z_current,
    linewidth=1.5,
    label="Current WAKIS",
)

ax.plot(
    f_reference / 1e9,
    Zabs_reference,
    linestyle="--",
    linewidth=1.0,
    alpha=0.6,
    marker="o",
    markersize=4,
    label="Previous regression data",
)

ax.axvline(
    f110 / 1e9,
    linestyle="--",
    linewidth=1.5,
    label=r"Analytical TM$_{110}$",
)

ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel(r"$|Z_\parallel|$ [$\Omega$]")
ax.set_title("CPML longitudinal impedance")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

## Relative change at the stored regression points

This plot shows where the previous regression samples differ most strongly from the current solution. Large pointwise differences near a shifted resonance are expected even if the spectral shape itself remains similar.

In [ ]:
relative_difference = (Z_sampled - Zabs_reference) / np.maximum(Zabs_reference, 1e-12)

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    f_reference / 1e9,
    100.0 * relative_difference,
    marker="o",
    markersize=4,
    linewidth=1.0,
)

ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

ax.axvline(
    f110 / 1e9,
    linestyle="--",
    linewidth=1.5,
    label=r"Analytical TM$_{110}$",
)

ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel("Difference [%]")
ax.set_title("Current vs. previous CPML impedance regression")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

## Numerical comparison summary

In [ ]:
abs_difference = np.abs(Z_sampled - Zabs_reference)

relative_abs_difference = abs_difference / np.maximum(Zabs_reference, 1e-12)

imax = np.argmax(relative_abs_difference)

tol_r = 50e-5
tol_a = 50e-5

close = np.isclose(
    Z_sampled,
    Zabs_reference,
    rtol=tol_r,
    atol=tol_a,
)

print(f"Number of sampled points:          {len(Z_sampled)}")
print(f"Maximum absolute difference:      {np.max(abs_difference):.6e} Ohm")
print(
    f"Maximum relative difference:      {100.0 * np.max(relative_abs_difference):.3f} %"
)
print(f"Frequency of max rel. difference: {f_reference[imax] / 1e9:.6f} GHz")
print(f"Regression points inside old tolerance: {np.sum(close)} / {len(close)}")

print("\nPoints outside old tolerance:")

for i in np.where(~close)[0]:
    print(
        f"{i:3d}: "
        f"f={f_reference[i] / 1e9:8.5f} GHz, "
        f"current={Z_sampled[i]:11.5e}, "
        f"reference={Zabs_reference[i]:11.5e}, "
        f"rel.diff={100.0 * relative_difference[i]:+8.2f} %"
    )

## Interpretation

The direct Gaussian-packet test checks the CPML independently of the cavity impedance regression. If both reflection criteria remain satisfied, the CPML continues to absorb correctly.

The wake/impedance comparison can nevertheless change because the transverse \(x\)- and \(y\)-boundaries are PEC. Correcting the high-side PEC discretisation changes the effective transverse electromagnetic domain and therefore shifts the resonant frequencies.

The analytical TM$_{110}$ reference provides an independent frequency scale for this shift. A change of the numerical resonance towards this analytical reference, while the direct CPML reflection criteria remain satisfied, indicates that the failing stored impedance regression is caused by the corrected boundary discretisation rather than by a degradation of the CPML.